# EDA 01: データ概要分析

ATMA Cup 20 (Udemy) の全データセットに対するEDA。

- **評価指標**: ROC-AUC
- **主キー**: `['社員番号', 'category']`
- **タスク**: 二値分類（target予測）

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
import seaborn as sns
from scipy import stats
from pathlib import Path

matplotlib.rcParams['font.family'] = 'IPAexGothic'

DATA_DIR = Path('../data/raw/input')
FIG_DIR = Path('../data/figures')
FIG_DIR.mkdir(parents=True, exist_ok=True)

## 1. データ読み込み

In [ ]:
df_train = pd.read_csv(DATA_DIR / 'train.csv')
df_test = pd.read_csv(DATA_DIR / 'test.csv')
df_udemy = pd.read_csv(DATA_DIR / 'udemy_activity.csv')
df_career = pd.read_csv(DATA_DIR / 'career.csv')
df_dx = pd.read_csv(DATA_DIR / 'dx.csv')
df_hr = pd.read_csv(DATA_DIR / 'hr.csv')
df_overtime = pd.read_csv(DATA_DIR / 'overtime_work_by_month.csv')
df_position = pd.read_csv(DATA_DIR / 'position_history.csv')

print('=== データサイズ ===')
for name, df in [('train', df_train), ('test', df_test), ('udemy_activity', df_udemy),
                  ('career', df_career), ('dx', df_dx), ('hr', df_hr),
                  ('overtime', df_overtime), ('position_history', df_position)]:
    print(f'{name}: {df.shape}')

## 2. train / test 基本確認

In [ ]:
print('=== train head ===')
display(df_train.head())
print('\n=== train dtypes ===')
display(df_train.dtypes)
print('\n=== train 欠損値 ===')
display(df_train.isnull().sum())
print('\n=== 重複行 ===')
print(f'train重複: {df_train.duplicated().sum()}')
print(f'test重複: {df_test.duplicated().sum()}')

In [ ]:
# ## 社員数確認
train_employees = set(df_train['社員番号'].unique())
test_employees = set(df_test['社員番号'].unique())
print(f'train社員数: {len(train_employees)}')
print(f'test社員数: {len(test_employees)}')
print(f'train/test重複社員数: {len(train_employees & test_employees)}')
print(f'trainのみ: {len(train_employees - test_employees)}')
print(f'testのみ: {len(test_employees - train_employees)}')

In [ ]:
# ## categoryの確認
train_cats = set(df_train['category'].unique())
test_cats = set(df_test['category'].unique())
print(f'train category数: {len(train_cats)}')
print(f'test category数: {len(test_cats)}')
print(f'train categories: {sorted(train_cats)}')
print(f'\ntrain/test category不一致: {train_cats.symmetric_difference(test_cats)}')

## 3. 目的変数の分析

In [ ]:
# ## 目的変数分布
print('=== target 分布 ===')
print(df_train['target'].value_counts())
print(f'\n正例率: {df_train["target"].mean():.4f}')

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# ## 全体の分布
df_train['target'].value_counts().plot(kind='bar', ax=axes[0], color=['steelblue', 'tomato'])
axes[0].set_title('target分布（全体）')
axes[0].set_xlabel('target')
axes[0].set_ylabel('count')

# ## categoryごとの正例率
category_target = df_train.groupby('category')['target'].mean().sort_values(ascending=False)
category_target.plot(kind='bar', ax=axes[1], color='steelblue')
axes[1].set_title('categoryごとの正例率')
axes[1].set_xlabel('category')
axes[1].set_ylabel('positive rate')
axes[1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.savefig(FIG_DIR / 'eda_01_target_dist.png', dpi=100, bbox_inches='tight')
plt.show()

print('\n=== categoryごとの詳細 ===')
display(df_train.groupby('category')['target'].agg(['count', 'sum', 'mean']).rename(columns={'count':'total','sum':'positive','mean':'positive_rate'}))

## 4. 各サブテーブルの基本確認

In [ ]:
# ## 各テーブルの基本情報を一括確認する関数
def summarize_df(name, df):
    print(f'\n=== {name} ===')
    print(f'shape: {df.shape}')
    print(f'社員数: {df["社員番号"].nunique()}')
    display(df.head(3))
    miss = df.isnull().sum()
    if miss.any():
        print('欠損値:')
        display(miss[miss > 0])
    else:
        print('欠損値: なし')
    print(f'重複行: {df.duplicated().sum()}')

for name, df in [('udemy_activity', df_udemy), ('career', df_career),
                  ('dx', df_dx), ('hr', df_hr),
                  ('overtime', df_overtime), ('position_history', df_position)]:
    summarize_df(name, df)

## 5. Udemy受講活動の分析

In [ ]:
# ## Udemy: 社員別の受講数集計
udemy_employees = set(df_udemy['社員番号'].unique())
print(f'Udemyデータに存在する社員数: {len(udemy_employees)}')
print(f'trainのうちUdemyデータあり: {len(train_employees & udemy_employees)}')
print(f'trainのうちUdemyデータなし: {len(train_employees - udemy_employees)}')

# ## 受講コース数の分布
udemy_per_employee = df_udemy.groupby('社員番号').size().reset_index(name='lecture_count')
print(f'\n社員あたりレクチャー数の統計:')
display(udemy_per_employee['lecture_count'].describe())

In [ ]:
# ## Udemyのカラム確認
display(df_udemy.dtypes)
print(df_udemy.columns.tolist())

In [ ]:
# ## コースカテゴリの確認
if 'コースカテゴリー' in df_udemy.columns:
    print('コースカテゴリー ユニーク数:', df_udemy['コースカテゴリー'].nunique())
    display(df_udemy['コースカテゴリー'].value_counts().head(20))

# ## マーク済み終了の確認
if 'マーク済み終了' in df_udemy.columns:
    print('\nマーク済み終了:')
    display(df_udemy['マーク済み終了'].value_counts())

In [ ]:
# ## Udemy活動量とtargetの関係
df_udemy_feat = df_udemy.groupby('社員番号').agg(
    lecture_count=('社員番号', 'count'),
).reset_index()

# ## trainとマージ
df_train_udemy = df_train.merge(df_udemy_feat, on='社員番号', how='left')
df_train_udemy['lecture_count'] = df_train_udemy['lecture_count'].fillna(0)

fig, ax = plt.subplots(figsize=(8, 4))
for t in [0, 1]:
    subset = df_train_udemy[df_train_udemy['target'] == t]['lecture_count']
    ax.hist(subset.clip(upper=200), bins=50, alpha=0.5, label=f'target={t}', density=True)
ax.set_title('Udemyレクチャー数分布 vs target')
ax.set_xlabel('lecture_count (clipped at 200)')
ax.set_ylabel('density')
ax.legend()
plt.tight_layout()
plt.savefig(FIG_DIR / 'eda_01_udemy_lecture_count.png', dpi=100, bbox_inches='tight')
plt.show()

## 6. 残業時間の分析

In [ ]:
display(df_overtime.head())
print('date dtype:', df_overtime['date'].dtype)
df_overtime['date'] = pd.to_datetime(df_overtime['date'])

# ## 月次残業時間の全体傾向
monthly_avg = df_overtime.groupby('date')['hours'].mean()
fig, ax = plt.subplots(figsize=(12, 4))
monthly_avg.plot(ax=ax)
ax.set_title('月別平均残業時間の推移')
ax.set_ylabel('平均残業時間')
plt.tight_layout()
plt.savefig(FIG_DIR / 'eda_01_overtime_trend.png', dpi=100, bbox_inches='tight')
plt.show()

print('\n残業時間の基本統計:')
display(df_overtime['hours'].describe())

In [ ]:
# ## 残業時間とtargetの関係
df_overtime_feat = df_overtime.groupby('社員番号')['hours'].agg(
    overtime_mean='mean',
    overtime_max='max',
    overtime_total='sum',
).reset_index()

df_train_ot = df_train.merge(df_overtime_feat, on='社員番号', how='left')

print('残業データの欠損率（train内）:')
print(df_train_ot[['overtime_mean', 'overtime_max', 'overtime_total']].isnull().mean())

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, col in zip(axes, ['overtime_mean', 'overtime_max', 'overtime_total']):
    for t in [0, 1]:
        subset = df_train_ot[df_train_ot['target'] == t][col].dropna()
        ax.hist(subset.clip(upper=subset.quantile(0.99)), bins=50, alpha=0.5, label=f'target={t}', density=True)
    ax.set_title(col)
    ax.legend()
plt.tight_layout()
plt.savefig(FIG_DIR / 'eda_01_overtime_vs_target.png', dpi=100, bbox_inches='tight')
plt.show()

## 7. 職位履歴の分析

In [ ]:
display(df_position.head())
print('\n役職ユニーク値:')
display(df_position['役職'].value_counts())
print('\n勤務区分ユニーク値:')
display(df_position['勤務区分'].value_counts())

In [ ]:
# ## 最新役職とtargetの関係
df_latest_pos = df_position.sort_values('year').groupby('社員番号').last().reset_index()
df_latest_pos = df_latest_pos.rename(columns={'役職': '最新役職', '勤務区分': '最新勤務区分'})

df_train_pos = df_train.merge(df_latest_pos[['社員番号', '最新役職', '最新勤務区分']], on='社員番号', how='left')

print('最新役職ごとの正例率:')
display(df_train_pos.groupby('最新役職')['target'].agg(['count', 'mean']).sort_values('mean', ascending=False))

## 8. DX研修・HR施策の分析

In [ ]:
display(df_dx.head())
print('\nDX研修カテゴリ:')
if '研修カテゴリ' in df_dx.columns:
    display(df_dx['研修カテゴリ'].value_counts())

# ## DX研修受講有無とtarget
dx_employees = set(df_dx['社員番号'].unique())
df_train['has_dx'] = df_train['社員番号'].isin(dx_employees).astype(int)
print('\nDX研修受講有無とtarget:')
display(df_train.groupby('has_dx')['target'].agg(['count', 'mean']))

In [ ]:
display(df_hr.head())
print('\nHRカテゴリ:')
if 'カテゴリ' in df_hr.columns:
    display(df_hr['カテゴリ'].value_counts())

# ## HR施策利用回数とtarget
df_hr_feat = df_hr.groupby('社員番号').size().reset_index(name='hr_count')
df_train_hr = df_train.merge(df_hr_feat, on='社員番号', how='left')
df_train_hr['hr_count'] = df_train_hr['hr_count'].fillna(0)
print('\nHR利用有無とtarget:')
df_train_hr['has_hr'] = (df_train_hr['hr_count'] > 0).astype(int)
display(df_train_hr.groupby('has_hr')['target'].agg(['count', 'mean']))

## 9. キャリア感アンケートの分析

In [ ]:
print('shape:', df_career.shape)
display(df_career.head())
print('\nカラム一覧:')
print(df_career.columns.tolist())
print('\n欠損値:')
display(df_career.isnull().sum())

## 10. まとめ・示唆

### 必要な前処理
- 残業時間・Udemy等のサブテーブルはtrain/testの`社員番号`でマージ。欠損（データなし）は0埋めか別フラグ化。
- `hr.csv`の実施日は単一日付と範囲カンマ区切りの2形式が混在 → パースロジック要作成。
- `position_history`のyearは下2桁（23→2023年）。

### 事実・示唆
- **主キーは`['社員番号', 'category']`**。1社員が複数categoryに対応する（ワイドフォーマット）。
- **目的変数の不均衡** → categoryごとに正例率が異なる可能性あり。categoryをグループ変数として扱う特徴量（相対的な学習量など）が有効な可能性。
- **Udemyデータなし社員が存在** → 受講フラグ自体が特徴量になりうる。
- **DX研修受講有無・HR施策利用数** → 学習意欲・昇進意欲の代理変数として有効な可能性。
- **残業時間** → 職種・時期により傾向が変わる可能性。月次粒度で時系列特徴量（直近N ヶ月平均等）を設計できる。
- **職位履歴** → 昇進速度（役職変化のペース）や最新役職レベルが強い特徴量になりうる。

### モデリング・特徴量設計への活用方針
- 社員単位の集約特徴量（Udemy受講数・完了率・残業時間統計・DX研修受講数・HR利用数）をベースに構築。
- categoryを固定効果として扱うか、category×社員の交差特徴量を作る。
- 時系列特徴量: 直近の活動量（直近3・6・12ヶ月）vs 全期間平均。
- キャリアアンケートの回答はNLP特徴量または順序符号化で活用。

### 追加で確認すべき観点
- キャリアアンケートの各質問列の分布と欠損パターン。
- Udemyの受講カテゴリと`category`（目的変数の分類）のクロス集計。
- 職位の上昇・降格パターンの分布。
- train/testで各サブテーブルのカバレッジ（結合率）の確認。